In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install torch==2.4.1 timm==1.0.9 tensorflow==2.17.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 788.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121

In [ ]:
import torch
import timm
from collections import OrderedDict

pth_path = "/content/drive/MyDrive/xception-b5690688.pth"  # path to your uploaded file

# Create same model architecture
model_torch = timm.create_model("xception", pretrained=False, num_classes=1)
state = torch.load(pth_path, map_location="cpu", weights_only=True)

# Handle different checkpoint formats
if "state_dict" in state:
    state = state["state_dict"]
# Remove 'module.' prefixes if saved from DataParallel
new_state = OrderedDict((k.replace("module.", ""), v) for k, v in state.items())

# Filter out layers with mismatched shapes
model_state_dict = model_torch.state_dict()
filtered_state_dict = OrderedDict()
for k, v in new_state.items():
    if k in model_state_dict and v.shape == model_state_dict[k].shape:
        filtered_state_dict[k] = v
    else:
        print(f"Skipping layer {k} due to size mismatch or not found in model.")


# Print keys and shapes from the loaded checkpoint and model for debugging
print("Checkpoint state_dict keys and shapes:")
for k, v in new_state.items():
    print(f"{k}: {v.shape}")

print("\nModel state_dict keys and shapes:")
for k, v in model_state_dict.items():
    print(f"{k}: {v.shape}")

# Load the filtered state dictionary
missing, unexpected = model_torch.load_state_dict(filtered_state_dict, strict=False)
print("Loaded PyTorch weights. Missing:", missing[:5], "Unexpected:", unexpected[:5])
model_torch.eval()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Skipping layer block1.rep.0.pointwise.weight due to size mismatch or not found in model.
Skipping layer block1.rep.3.pointwise.weight due to size mismatch or not found in model.
Skipping layer block2.rep.1.pointwise.weight due to size mismatch or not found in model.
Skipping layer block2.rep.4.pointwise.weight due to size mismatch or not found in model.
Skipping layer block3.rep.1.pointwise.weight due to size mismatch or not found in model.
Skipping layer block3.rep.4.pointwise.weight due to size mismatch or not found in model.
Skipping layer block4.rep.1.pointwise.weight due to size mismatch or not found in model.
Skipping layer block4.rep.4.pointwise.weight due to size mismatch or not found in model.
Skipping layer block4.rep.7.pointwise.weight due to size mismatch or not found in model.
Skipping layer block5.rep.1.pointwise.weight due to size mismatch or not found in model.
Skipping layer block5.rep.4.pointwise.weight due to size mismatch or not found in model.
Skipping layer block5

Xception(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act2): ReLU(inplace=True)
  (block1): Block(
    (skip): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
    (skipbn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (rep): Sequential(
      (0): SeparableConv2d(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): SeparableConv2d(
        (conv1): Conv

Now that the PyTorch weights are loaded, we can convert the model to TensorFlow. We will extract the weights from the loaded PyTorch model and then create a TensorFlow model with the same architecture and load the weights into it.

In [ ]:
import tensorflow as tf

# Get the state dictionary from the PyTorch model
torch_state_dict = model_torch.state_dict()

# Create a TensorFlow model with the same architecture (this requires defining the TensorFlow model)
# For demonstration, let's assume you have a function to create the TensorFlow model
# def create_tensorflow_model():
#     # Define and return your TensorFlow model here
#     pass

# tf_model = create_tensorflow_model()

# Assuming a simple mapping for demonstration purposes.
# In a real scenario, you would need a more sophisticated mapping
# based on the layer names and structures of both models.
# This is a placeholder and will likely need to be adapted
# based on the specific TensorFlow model definition.

# Placeholder for TensorFlow model creation - Replace with your actual TensorFlow model definition
# Example:
# tf_model = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(...)
#     # ... other layers matching the PyTorch model
# ])

# Since we don't have the TensorFlow model definition, we'll just print the PyTorch weights
# that would need to be mapped to the TensorFlow model.
print("PyTorch weights extracted for TensorFlow conversion:")
for name, param in torch_state_dict.items():
    print(f"{name}: {param.shape}")

# In a real conversion, you would map 'name' to the corresponding TensorFlow layer name
# and load the numpy array (param.detach().cpu().numpy()) into the TensorFlow layer.
# Example (requires a defined tf_model):
# tf_layer = tf_model.get_layer(tf_layer_name)
# tf_layer.set_weights([param.detach().cpu().numpy()])

PyTorch weights extracted for TensorFlow conversion:
conv1.weight: torch.Size([32, 3, 3, 3])
bn1.weight: torch.Size([32])
bn1.bias: torch.Size([32])
bn1.running_mean: torch.Size([32])
bn1.running_var: torch.Size([32])
bn1.num_batches_tracked: torch.Size([])
conv2.weight: torch.Size([64, 32, 3, 3])
bn2.weight: torch.Size([64])
bn2.bias: torch.Size([64])
bn2.running_mean: torch.Size([64])
bn2.running_var: torch.Size([64])
bn2.num_batches_tracked: torch.Size([])
block1.skip.weight: torch.Size([128, 64, 1, 1])
block1.skipbn.weight: torch.Size([128])
block1.skipbn.bias: torch.Size([128])
block1.skipbn.running_mean: torch.Size([128])
block1.skipbn.running_var: torch.Size([128])
block1.skipbn.num_batches_tracked: torch.Size([])
block1.rep.0.conv1.weight: torch.Size([64, 1, 3, 3])
block1.rep.0.pointwise.weight: torch.Size([128, 64, 1, 1])
block1.rep.1.weight: torch.Size([128])
block1.rep.1.bias: torch.Size([128])
block1.rep.1.running_mean: torch.Size([128])
block1.rep.1.running_var: torch.Size

In [ ]:
import numpy as np

torch_weights = {}
for name, param in model_torch.state_dict().items():
    torch_weights[name] = param.detach().cpu().numpy()
print(f"Extracted {len(torch_weights)} weight tensors from PyTorch checkpoint.")


Extracted 276 weight tensors from PyTorch checkpoint.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import Xception

keras_model = tf.keras.applications.Xception(
    include_top=False, weights=None, input_shape=(299, 299, 3)
)
x = layers.GlobalAveragePooling2D()(keras_model.output)
out = layers.Dense(1, activation="sigmoid")(x)
model_keras = models.Model(keras_model.input, out)


In [ ]:
keras_layers = {l.name: l for l in model_keras.layers if l.weights}

transferred = 0
for name, arr in torch_weights.items():
    for lname, layer in keras_layers.items():
        if lname in name and len(layer.get_weights()) == 1 and arr.ndim == 4:
            # conv weights: transpose (PyTorch: [out, in, h, w] → Keras: [h, w, in, out])
            w = np.transpose(arr, (2, 3, 1, 0))
            try:
                layer.set_weights([w])
                transferred += 1
            except Exception:
                pass

print(f"Transferred {transferred} conv weights (partial init).")


Transferred 0 conv weights (partial init).


In [ ]:
h5_path = "/content/xception_ffpp_weights.h5"
model_keras.save(h5_path)
print("Saved:", h5_path)


Saved: /content/drive/MyDrive/xception_ffpp_weights.h5
